# Sensitivity of susceptible/protected noise attenuation

This notebook tests

$$
c_{SH}\in\{0,\;0.05,\;0.1,\;0.2,\;0.5,\;1\}.
$$

The factor $c_{SH}$ multiplies only the Wiener noise in the large susceptible ($S$) and protected ($H$) compartments. It does **not** change the deterministic compartment flows and it is not treated as a biological transmission parameter.

The comparison is leakage-safe. A single Poisson calibration is fitted using only the first 60 weekly observations. Those mean-dynamics parameters are frozen. Each $c_{SH}$ value is then assessed at the same balanced set of later, non-overlapping six-week forecast origins. Recent observations available at each origin may initialise hidden states, but the held-out six-week outcomes are not used to fit or condition that forecast.

The final candidate is $c_{SH}=1$, which removes the additional attenuation and uses $\omega S\,dW^S$ and $\omega H\,dW^H$ directly. The experiment checks whether conclusions are robust to smaller values and whether the unattenuated setting damages held-out interval coverage or outbreak-probability accuracy. Setting $c_{SH}=0$ removes noise only from $S$ and $H$; $E$, $I$, $Q$, and $D$ remain stochastic.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if repo_root.name == 'outbreak_probability_model':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from outbreak_probability_model import load_default_inputs
from outbreak_probability_model.london_calibration import (
    CalibrationConfig,
    HistoryConditioningConfig,
    SEASONAL_FIT_PARAMETER_BOUNDS,
    default_calibration_parameters,
    load_london_confirmed_cases,
    make_balanced_blocked_validation_design,
    strict_forecast_validation,
)

output_dir = repo_root / 'experiments' / 'measles' / 'London' / 'sh_noise_sensitivity'
strict_fit_path = (repo_root / 'experiments' / 'measles' / 'London' /
                   'calibration_strict_validation_final' /
                   '03_training_fit_and_origin_states.csv')
output_dir.mkdir(parents=True, exist_ok=True)
print('Outputs:', output_dir)

## Experiment settings

`quick_check=False` is the intended analysis. It uses 400 stochastic trajectories for each selected historical origin and each of six values of $c_{SH}$. This can take time because every trajectory solves the regional-age SDE at a 0.02-day time step.

Set `quick_check=True` only to verify that the notebook runs; do not report its small-sample results.

In [ ]:
quick_check = False
c_sh_levels = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
training_weeks = 60
horizon_weeks = 6
outbreak_threshold = 15.0

if quick_check:
    calibration_config = CalibrationConfig(
        block_weeks=6, n_trials=1, n_refinement_trials=0,
        initial_state_refinement_maxiter=0, warmup_weeks=0,
        random_seed=20260818,
    )
    n_simulations = 5
else:
    calibration_config = CalibrationConfig(
        block_weeks=6, n_trials=1, n_refinement_trials=0,
        initial_state_refinement_maxiter=0, seasonal_refinement_maxiter=0,
        warmup_weeks=0,
        random_seed=20260818,
    )
    n_simulations = 400

history_conditioning = HistoryConditioningConfig(
    history_weeks=4,
    transmission_multiplier_bounds=(0.8, 1.25),
    regularization_strength=1.0,
    origin_observation_weight=4.0,
    maxiter=80,
)

settings = pd.DataFrame([{
    'quick_check': quick_check,
    'training_weeks': training_weeks,
    'horizon_weeks': horizon_weeks,
    'outbreak_threshold': outbreak_threshold,
    'n_simulations_per_origin_and_level': n_simulations,
    'gamma_lower_bound': SEASONAL_FIT_PARAMETER_BOUNDS['gamma'][0],
    'gamma_upper_bound': SEASONAL_FIT_PARAMETER_BOUNDS['gamma'][1],
    'c_sh_levels': ', '.join(map(str, c_sh_levels)),
    'frozen_parameter_source': str(strict_fit_path),
}])
display(settings)
settings.to_csv(output_dir / '00_settings.csv', index=False)

## Choose held-out forecast origins

After the initial 60-week calibration period, the remaining observations are divided into non-overlapping six-week outcome blocks. The helper retains all blocks from the smaller outcome class and a time-stratified sample of the same number from the other class. Selection uses only the observed outbreak labels and a fixed seed, before inspecting any model forecasts.

In [ ]:
cases = load_london_confirmed_cases()
inputs = load_default_inputs()
base_parameters = default_calibration_parameters()

design = make_balanced_blocked_validation_design(
    cases,
    training_weeks=training_weeks,
    horizon_weeks=horizon_weeks,
    outbreak_threshold=outbreak_threshold,
    random_seed=20260818,
)
selected_design = design.query('selected_for_balanced_test').copy()
cutoff_indices = selected_design['cutoff_index'].astype(int).tolist()
design.to_csv(output_dir / '01_validation_design.csv', index=False)
display(selected_design)
print('Selected forecast origins:', cutoff_indices)

## Load the frozen training-only seasonal dynamics

The strict validation notebook has already fitted the seasonal Poisson vector using only the first 60 observations. This notebook reuses that exact frozen vector for every historical origin and every $c_{SH}$ level. Reusing it avoids a redundant optimisation and guarantees that only the stochastic attenuation factor changes.

In [ ]:
if not strict_fit_path.exists():
    raise FileNotFoundError(
        'Run London_Strict_Forecast_Validation.ipynb first; missing ' +
        str(strict_fit_path)
    )
strict_fits = pd.read_csv(strict_fit_path)
seasonal_fits = strict_fits.query("model == 'seasonal'").copy()
if seasonal_fits.empty:
    raise RuntimeError('The strict validation output has no seasonal training fit.')
if any(seasonal_fits[name].nunique(dropna=False) != 1
       for name in SEASONAL_FIT_PARAMETER_BOUNDS):
    raise RuntimeError('Seasonal global parameters were not frozen across strict origins.')
frozen_vector = {name: float(seasonal_fits.iloc[0][name])
                 for name in SEASONAL_FIT_PARAMETER_BOUNDS}
pd.DataFrame([frozen_vector]).to_csv(
    output_dir / '02_frozen_poisson_parameters.csv', index=False
)
display(pd.DataFrame({'parameter': frozen_vector.keys(), 'value': frozen_vector.values()}))

## Run the $c_{SH}$ sensitivity forecasts

All scenarios use identical cutoffs, fitted mean parameters, history conditioning, random-seed scheme, reporting model, and total noise scale $\omega$. Only $c_{SH}$ changes.

In [ ]:
all_summary = []
all_paths = []
all_origins = []

for c_sh in c_sh_levels:
    print(f'Running c_SH={c_sh:g} ...')
    scenario_base = replace(base_parameters, sh_noise_multiplier=float(c_sh))
    summary, paths, origins = strict_forecast_validation(
        cases,
        inputs=inputs,
        config=calibration_config,
        cutoff_indices=cutoff_indices,
        horizon_weeks=horizon_weeks,
        n_stochastic_simulations=n_simulations,
        outbreak_threshold=outbreak_threshold,
        base_parameters=scenario_base,
        history_conditioning=history_conditioning,
        refit_parameters_each_cutoff=False,
        objective_metric='poisson_nll',
        fitted_vector_override=frozen_vector,
        parameter_bounds=SEASONAL_FIT_PARAMETER_BOUNDS,
        progress=False,
    )
    for frame in (summary, paths, origins):
        frame.insert(0, 'c_sh', float(c_sh))
    all_summary.append(summary)
    all_paths.append(paths)
    all_origins.append(origins)

summary = pd.concat(all_summary, ignore_index=True)
paths = pd.concat(all_paths, ignore_index=True)
origins = pd.concat(all_origins, ignore_index=True)
summary.to_csv(output_dir / '05_forecast_summary.csv', index=False)
paths.to_csv(output_dir / '06_forecast_paths.csv', index=False)
origins.to_csv(output_dir / '07_origin_probability_scores.csv', index=False)
print('Completed', len(paths), 'weekly simulated path rows.')

## Compare forecast performance

The target p10--p90 interval has nominal 80% coverage. Coverage alone is not enough: an extremely wide interval can cover most observations while being uninformative. Therefore the table reports coverage together with interval width, median forecast error, and Brier score for the six-week outbreak probability. Lower MAE, RMSE, Brier score, interval width, and absolute coverage gap are better, but these criteria should be interpreted jointly rather than collapsed into a newly invented single score.

In [ ]:
metric_rows = []
for c_sh, group in summary.groupby('c_sh', sort=True):
    residual = group['median_cases'] - group['observed_cases']
    origin_group = origins[origins['c_sh'].eq(c_sh)]
    covered = group['observed_cases'].between(group['p10_cases'], group['p90_cases'])
    metric_rows.append({
        'c_sh': c_sh,
        'primary_unattenuated_setting': bool(np.isclose(c_sh, 1.0)),
        'forecast_weeks_scored': len(group),
        'forecast_origins_scored': group['cutoff_id'].nunique(),
        'median_forecast_MAE': float(np.abs(residual).mean()),
        'median_forecast_RMSE': float(np.sqrt(np.mean(residual ** 2))),
        'p10_p90_coverage': float(covered.mean()),
        'coverage_gap_from_80_percent': float(abs(covered.mean() - 0.8)),
        'mean_p10_p90_width': float((group['p90_cases'] - group['p10_cases']).mean()),
        'mean_outbreak_probability_Brier_score': float(origin_group['event_brier_score'].mean()),
    })

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(output_dir / '08_c_sh_metric_comparison.csv', index=False)
display(metrics.style.format({
    'median_forecast_MAE': '{:.3f}',
    'median_forecast_RMSE': '{:.3f}',
    'p10_p90_coverage': '{:.1%}',
    'coverage_gap_from_80_percent': '{:.1%}',
    'mean_p10_p90_width': '{:.3f}',
    'mean_outbreak_probability_Brier_score': '{:.4f}',
}))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5), sharex=True)
plots = [
    ('median_forecast_MAE', 'Median forecast MAE', False),
    ('p10_p90_coverage', 'p10--p90 coverage', True),
    ('mean_p10_p90_width', 'Mean p10--p90 width', False),
    ('mean_outbreak_probability_Brier_score', 'Mean Brier score', False),
]
for ax, (column, label, show_target) in zip(axes.flat, plots):
    ax.plot(metrics['c_sh'], metrics[column], 'o-', color='#286090')
    ax.axvline(1.0, color='#d95f02', ls='--', lw=1.2, label='unattenuated 1.0')
    if show_target:
        ax.axhline(0.8, color='black', ls=':', lw=1, label='nominal 80%')
    ax.set_ylabel(label)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
for ax in axes[-1, :]:
    ax.set_xlabel(r'$c_{SH}$')
fig.suptitle(r'Held-out sensitivity to $S/H$ noise attenuation')
fig.tight_layout()
fig.savefig(output_dir / '09_c_sh_metric_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

## How to interpret the result

- If the main forecast conclusions and Brier scores are similar across the grid, the result is robust and using the simpler unattenuated form $c_{SH}=1$ is reasonable.
- If $c_{SH}=1$ has very poor coverage or Brier score relative to smaller values, report that and retain an attenuation factor supported by the validation evidence.
- Do not select a large value solely because it gives wider intervals. Good uncertainty should approach the nominal 80% coverage without making intervals unnecessarily wide, and probability forecasts should retain a low Brier score.
- Keep the full sensitivity table in the Results chapter or appendix; the Methods section should describe the grid and validation design, not claim the winning value before this notebook has run.